# Fine-Tune LLM with Data Cache

This notebook demonstrates how to fine-tune a BERT model with Kubeflow TrainJob using data streaming from a distributed cache.

We will use an iceberg table with [the Amazon reviews dataset](https://huggingface.co/datasets/fancyzhx/amazon_polarity) loaded from data cache cluster, which provides efficient data streaming for distributed training workloads.

## Install Kubeflow SDK

In [30]:
# Install the Kubeflow SDK
!pip install -U kubeflow

## List Available Training Runtimes

Get available Kubeflow Training Runtimes using the list_runtimes() API.

In [31]:
from kubeflow.trainer import TrainerClient

client = TrainerClient()
for runtime in client.list_runtimes():
    print(f'{runtime}\n')

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.2.1', num_nodes=1, device='Unknown', device_count='1'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

Runtime(name='jax-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='jax', image='nvcr.io/nvidia/jax:25.10-py3', num_nodes=1, device='Unknown', device_count='Unknown'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.2.1', num_nodes=1, device='Unknown', device_count='1'), kind=<RuntimeKind.CLUSTER_TRAINING_RUNTIME: 'ClusterTrainingRuntime'>, pretrained_model=None)

R

## Define Training Function with DataCacheDataset

This training function uses DataCacheDataset to stream data from the cache cluster.

In [32]:
def train_func():
    import time
    import datetime
    from multiprocessing import current_process
    import torch.distributed as dist
    import pyarrow as pa
    import torch
    import os
    from torch.utils.data import IterableDataset
    from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, AdamW

    # Low-memory stub for local CPU training
    class StubAmazonReviewDataset(IterableDataset):
        def __init__(self, tokenizer, seed=0, num_samples=16, batch_size=2):
            super().__init__()
            self.tokenizer = tokenizer
            self.seed = seed
            self.num_samples = num_samples
            self.batch_size = batch_size
            self.row_count = 0

            self.sample_texts = [
                "Great product, works perfectly!",
                "Terrible quality, broke quickly.",
            ] * (num_samples // 2)
            self.sample_labels = [1, 0] * (num_samples // 2)

        def set_epoch(self, epoch):
            self.seed = self.seed + epoch

        def __iter__(self):
            for i in range(0, self.num_samples, self.batch_size):
                batch_texts = self.sample_texts[i:i + self.batch_size]
                batch_labels = self.sample_labels[i:i + self.batch_size]
                self.row_count += len(batch_texts)

                encoding = self.tokenizer(
                    batch_texts,
                    padding="max_length",
                    truncation=True,
                    max_length=32,  # Short sequence length saves significant memory
                    return_tensors="pt"
                )

                yield {
                    "input_ids": encoding["input_ids"],
                    "attention_mask": encoding["attention_mask"],
                    "labels": torch.tensor(batch_labels, dtype=torch.long)
                }

    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )
    device = torch.device(f"{device}:{local_rank}")
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

    start = time.time()
    dataset = StubAmazonReviewDataset(tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=None)

    for epoch in range(1):
        dataset.set_epoch(epoch)
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            if dist.get_rank() == 0:
                print(f"Epoch {epoch}, Loss: {loss.item()}")

    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    end = time.time()
    print(f"Time taken: {end - start:.6f} seconds")
    print(f"Total rows processed: {dataset.row_count}")

    dist.destroy_process_group()

## Create TrainJob with Data Cache

Use the train() API with DataCacheInitializer to create a TrainJob that streams data from cache.

In [33]:
from kubeflow.trainer import CustomTrainer

selected_runtime = None
for runtime in client.list_runtimes():
    if runtime.name in ["torch-distributed", "torch-distributed-with-cache"]:
        selected_runtime = runtime
        break

job_name = client.train(
    trainer=CustomTrainer(
        func=train_func,
        num_nodes=1,
        resources_per_node={
            "cpu": "1000m",
            "memory": "3Gi",
        },
        packages_to_install=["pyarrow==19.0.0", "transformers==4.45.2"]
    ),
    runtime=selected_runtime
)

print(f"TrainJob created: {job_name}")

TrainJob created: p5be3f9c889b


## Check the TrainJob Info

Use the list_jobs() and get_job() APIs to get information about the created TrainJob.

In [34]:
for job in client.list_jobs():
    print(f"TrainJob: {job.name}, Status: {job.status}, Created at: {job.creation_timestamp}")

TrainJob: p5be3f9c889b, Status: Running, Created at: 2026-10-19 04:37:34+00:00
TrainJob: q08907506260, Status: Complete, Created at: 2026-10-19 04:33:39+00:00


In [35]:
# Get details about the TrainJob steps
for step in client.get_job(name=job_name).steps:
    print(f"Step: {step.name}, Status: {step.status}, Devices: {step.device} x {step.device_count}\n")

Step: node-0, Status: Running, Devices: cpu x 1



## Watch the TrainJob Logs

Use the get_job_logs() API to follow the TrainJob logs in real-time.

In [36]:
# Wait for the job to reach running state
client.wait_for_job_status(job_name, status={"Running"})
print("Job is in running state")

# Follow the logs
for line in client.get_job_logs(job_name, follow=True):
    print(line)

Job is in running state
Using Device: cpu, Backend: gloo
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
Distributed Training for WORLD_SIZE: 1, RANK: 0, LOCAL_RANK: 0
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/root/.local/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 0, Loss: 0.6917177438735962
Epoch 0, Loss: 0.6609763503074646
Epoch 0, Loss: 0.6315245628356934
Epoch 0, Loss: 0.596214

## Verify TrainJob Completion

Check the TrainJob status to ensure it completed successfully.

In [37]:
client.wait_for_job_status(name=job_name, timeout=300)
print(f"TrainJob {job_name} finished successfully!")

TrainJob p5be3f9c889b finished successfully!


## Delete the TrainJob

When the TrainJob is finished, you can delete the resource.

In [39]:
# Uncomment to delete the job
client.delete_job(job_name)
print(f"TrainJob {job_name} deleted successfully.")

TrainJob p5be3f9c889b deleted successfully.
